# Phase 1 — Data Pipeline: Jumia Nigeria Reviews
### DSN × BCT LLM Agent Hackathon

This notebook completes **Phase 1** end-to-end for our new User-Centric Jumia dataset:

1. **Scraping**: We use a custom scraper to collect authentic product reviews from Jumia Nigeria.
2. **User Profiles**: We invert the product-centric data to build user profiles, focusing on capturing the "Nigerian Voice".
3. **Exploratory Data Analysis**: We analyze the distribution of reviews and word counts.
4. **Qdrant Indexing**: We embed user profiles and index them into Qdrant for semantic search and style retrieval.
5. **Verification**: We run test queries to ensure the vector DB behaves as expected.


## 0. Environment Setup

In [1]:
# Install / upgrade dependencies (run once)
%pip install -q datasets sentence-transformers qdrant-client tqdm matplotlib seaborn wordcloud pandas numpy bs4 requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import statistics
from pathlib import Path

import matplotlib.pyplot as plt
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 4)})
print("All imports OK ✓")

/home/alli-ekundayo/Projects/Ego/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports OK ✓


## 1. Data Collection: Scraping Jumia
Instead of the generic Amazon dataset, we scrape authentic reviews from Jumia Nigeria to capture real Nigerian linguistic patterns and accents. We run our scraper across multiple categories.

In [3]:
# Run the scraper (Note: We've already run a large scrape in the background,
# so you can skip this if data/jumia_reviews.json exists)
!python ../scripts/scrape_jumia.py --categories phones-tablets electronics computing health-beauty home-office fashion groceries mlp-appliances baby-products video-games books-movies-music sporting-goods --limit 100000 --pages 15 --review-pages 5 --delay 1.0

2026-05-08 15:33:46 [INFO] 
── Category: Phones & Tablets ──  (pages=1–15)
2026-05-08 15:33:46 [INFO]   Listing page 1/15 → https://www.jumia.com.ng/phones-tablets/
2026-05-08 15:33:49 [INFO]   No 'Next Page' link — reached last page at page 1.
2026-05-08 15:33:49 [INFO]   Total unique products found across listing pages: 57
2026-05-08 15:33:49 [INFO]   ↳ Tecno  Camon 50 Pro 8GB/ 256GB Android 16 - Black
2026-05-08 15:33:50 [INFO]     → Reviews: https://www.jumia.com.ng/catalog/productratingsreviews/sku/TE339MP7W55ONNAFAMZ/
2026-05-08 15:33:51 [INFO]     (no reviews — skipping)
2026-05-08 15:33:51 [INFO]   ↳ itel A200+  AI Fusion Camera System 128gb/3+5gb 6000mah IP65 120Hz 6.75" Android  Titanium + LIMITED GIFT A1112
2026-05-08 15:33:52 [INFO]     → Reviews: https://www.jumia.com.ng/catalog/productratingsreviews/sku/IT724MP82HRGVNAFAMZ/
2026-05-08 15:33:52 [INFO]     (no reviews — skipping)
2026-05-08 15:33:53 [INFO]   ↳ Samsung Galaxy A36 6.7" 6GB RAM/128GB ROM Android 15 - Lime Gree

## 2. Building User Profiles
The Ego user modeling agent needs user-centric data. We invert the scraped product reviews into user profiles. Each profile aggregates all reviews by a single user, and the `voice_sample` feature concatenates their text for embedding.

In [4]:
# Build profiles and filter for users with enough reviews
!python ../scripts/build_user_profiles.py --min-reviews 5

2026-05-08 16:39:59 [INFO] Loaded 369 products (4200 total reviews) from data/jumia_reviews.json
2026-05-08 16:39:59 [INFO] Found 2775 unique reviewers in total
2026-05-08 16:39:59 [INFO] 
── Review-count distribution across 2775 users ──
2026-05-08 16:39:59 [INFO]   ≥ 1 reviews: 2775 users  ███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████

## 3. Exploratory Data Analysis (EDA)
Let's analyze the `user_profiles.json` dataset to understand review lengths and user distributions. This directly informs our Agent's voice capabilities.

In [9]:
DATA_DIR = Path("../data")
with open(DATA_DIR / "user_profiles.json", "r", encoding="utf-8") as f:
    users = json.load(f)

all_bodies = []
for user in users:
    all_user_reviews = user.get("train_reviews", []) + user.get("test_reviews", [])
    for r in all_user_reviews:
        body = (r.get("body") or "").strip()
        all_bodies.append(body)

word_counts = [len(b.split()) for b in all_bodies]
non_empty = [w for w in word_counts if w > 0]

print(f"Total reviews      : {len(all_bodies)}")
print(f"Empty bodies       : {sum(1 for w in word_counts if w == 0)}")

if non_empty:
    print(f"≤ 3 words          : {sum(1 for w in non_empty if w <= 3)}")
    print(f"4 – 10 words       : {sum(1 for w in non_empty if 4 <= w <= 10)}")
    print(f"11 – 30 words      : {sum(1 for w in non_empty if 11 <= w <= 30)}")
    print(f"> 30 words         : {sum(1 for w in non_empty if w > 30)}")
    print(f"Median length      : {statistics.median(non_empty):.0f} words")
    print(f"Mean length        : {statistics.mean(non_empty):.1f} words")

    # Show the longest 5 unique reviews
    unique_bodies = list(set(all_bodies))
    longest = sorted(unique_bodies, key=lambda b: len(b.split()), reverse=True)[:5]
    print("\n--- 5 longest unique review bodies ---")
    for i, b in enumerate(longest, 1):
        print(f"\n{i}. ({len(b.split())} words)\n   {b}")
else:
    print("No non-empty reviews found.")

Total reviews      : 228
Empty bodies       : 0
≤ 3 words          : 78
4 – 10 words       : 101
11 – 30 words      : 42
> 30 words         : 7
Median length      : 5 words
Mean length        : 7.7 words

--- 5 longest unique review bodies ---

1. (56 words)
   It just like aloe vera gel, Best at night or indoor use only. Feels more sticky in the afternoon especially if you sweat alot, And it doesn't go well with sunscreen It irritates my face but it gradually brighten up the face and it lightweight and absorb well. i used for 4 days before Discontinue use.

2. (50 words)
   A paper showing how to arrange the chair, will go a long way, can't believe they couldn't print a paper with the pictorial guide for the arrangement of the chair. Also if the foam could be a little bit more comfy, it would go a long way. Hopefully it lasts.

3. (49 words)
   So far so good, it’s working perfectly fine it’s a small surface so I can fit more than two Standard size pots at a time it can however fit sm

## 4. Qdrant Setup & Indexing
We embed the `voice_sample` for each user using `all-MiniLM-L6-v2` and index it in Qdrant under the `user_profiles` collection.

In [10]:
# Run the indexing script to embed and upsert into Qdrant
!PYTHONPATH=.. python ../data/build_index.py --collection user_profiles

/home/alli-ekundayo/Projects/Ego/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Loading weights: 100%|██████████████████████| 103/103 [00:00<00:00, 4051.10it/s]
2026-05-08 16:51:36,700 [INFO] Creating / recreating Qdrant collection 'user_profiles' (dim=384)…
2026-05-08 16:51:37,091 [INFO] HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2026-05-08 16:51:37,650 [INFO] HTTP Request: DELETE http://localhost:6333/collections/user_profiles "HTTP/1.1 200 OK"
2026-05-08 16:51:41,066 [INFO] HTTP Request

## 5. Verification Queries
Let's test our Qdrant index to retrieve similar user profiles based on a query text.

In [13]:
client = QdrantClient("localhost", port=6333)
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

query = "I bought this for my mum and she loves it"
query_vec = encoder.encode([query], normalize_embeddings=True)[0].tolist()

hits = client.query_points(
    collection_name="user_profiles", query=query_vec, limit=3, with_payload=True
)

print(f"=== Semantic user search: '{query}' ===")
for i, h in enumerate(hits.points, 1):
    name = h.payload.get("name", "?")
    reviews_count = h.payload.get("review_count", 0)
    score = round(h.score, 4)
    print(f"  {i}. [{score}] User: {name} (Total reviews: {reviews_count})")

    # Show a sample of their reviews
    sample_reviews = h.payload.get("sample_reviews", [])
    if sample_reviews:
        print(f"      Sample voice: {sample_reviews[0][:150]}...")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2790.12it/s]


=== Semantic user search: 'I bought this for my mum and she loves it' ===
  1. [0.3958] User: Rebecca (Total reviews: 5)
      Sample voice: I love it, freezes fast, less noise and energy consumption...
  2. [0.3759] User: Ifeoluwa (Total reviews: 6)
      Sample voice: I love

A bit smaller in size.......
  3. [0.3746] User: Rita (Total reviews: 5)
      Sample voice: Good quality...


## 6. Next Steps
With the core Data Pipeline shifted to **Jumia User Profiles**, the vector database now holds authentic Nigerian vernacular data. 

Next, proceed to `02_task_a_modeling.ipynb` or the relevant Agent notebook to leverage these profiles for style transfer and rating prediction.